# Biopython으로 폐암 데이터 받아서 분석해보기 
## 개요
암... 현대인에게는 여전히 풀어야 할 숙제입니다. 물론 조기에 진단을 받으면 살 수도 있지만, 때가 늦어지면... 아... 의사쌤이 뭔가 심각한 얼굴로 부르기 시작하는데... 

그거 아십니까? 암은 다양한 요인들에 의해 발생하고 그 중 하나가 유전자 변이입니다. 옹코진(Oncogene-종양 유전자)이라는 놈이 있는데 이놈은 변이가 돼서 미쳐 날뛰는 순간 암이 되는거고, 암 억제 유전자는 변이가 터져서 제 기능을 못 하게 되면 암으로 발전하게 됩니다. 물론 우리 몸은 신호체계로 돌아가는거라 어떻게든 막고 막고 막는 경로가 존재는 하겠지만, 그 경로가 다 뻑나면... 아... 

그러니까 여러분은 담배를 멀리하시고 건강한 삶을 사시는 게 좋습니다. 돌연변이원 중 하나가 담배임. 

## 결론
**담배 끊으십쇼.**

## 프로젝트 정보
- 인원: 1인(개인 프로젝트)
- 버전: 3.10(TF_base)
- 설치할 것들: Biopython, lifelines(생존곡선 잘그려줌), GEOparse(NCBI GEO에 접근할 때 필요)
- 데이터 리소스: NCBI(Entrez로 갖고올 예정)

In [ ]:
# 모듈
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from lifelines import KaplanMeierFitter # 이친구가 생존곡선을 잘 그려요 아무튼 그럼 
import GEOparse # NCBI GEO에 접근할 때 필요함 

from Bio import Entrez # NCBI 창고털이 드가자 

# 통계분석용
from scipy import stats
from scipy.stats import kruskal
from scipy.stats import mannwhitneyu
from scipy.stats import shapiro
from scipy.stats import levene
import scikit_posthocs as sp

# 그래프를 그리기 위한 기본 설정
plt.rcParams['font.family'] = 'Nanumsquare_ac'
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# NCBI 창고를 털려면 이메일이 필요함 
Entrez.email = "blackholekun@gmail.com" # 이메일

# 데이터를 털어라! 

In [ ]:
# 창고는 열려있다! 데이터를 털어라! 
print("--- GSE30219 데이터 다운로드 중... ---")
gse_mut = GEOparse.get_GEO(geo="GSE30219", destdir="./") # 진짜 창고 터는중 
df_mut = gse_mut.phenotype_data

In [ ]:
# 정보 확인
df_mut.info() # 거 정보좀 봅시다. 
df_mut.isna().sum() # 아... 널이 있었어... 
df_mut.head()

# NSCLC, SCLC 생존 곡선

In [ ]:
# 필요한 것만 쏙 빼오기 
df_target = df_mut[[
    'characteristics_ch1.3.histology', 
    'characteristics_ch1.7.follow-up time (months)', 
    'characteristics_ch1.8.status'
]].copy()
# Histology: 암종
# Months: 생존기간
# Status: 생존 상태
df_target.columns = ['Histology', 'Months', 'Status'] # 컬럼 이름 바꿀거임 

# 선가공 
df_target['Months'] = pd.to_numeric(df_target['Months'], errors='coerce')
# 돌아가심->1, 살아있음->0
df_target['Event'] = df_target['Status'].apply(lambda x: 1 if str(x).upper() == 'DEAD' else 0)

# 암종 크게 분류 (NSCLC vs SCLC)
# NSCLC: 비소세포성 폐암
# SCLC: 소세포성 폐암
def classify_lung_cancer(x):
    if x in ['ADC', 'SQC', 'LCC', 'LCNE', 'BAS']: return 'NSCLC (Non-Small Cell)'
    elif x == 'SCC': return 'SCLC (Small Cell)'
    elif x == 'NTL': return 'Normal/Control'
    else: return 'Other'

df_target['Group'] = df_target['Histology'].apply(classify_lung_cancer)

# 생존 분석 시각화
kmf = KaplanMeierFitter()
plt.figure(figsize=(10, 6))

for name, grouped_df in df_target.groupby('Group'):
    if name == 'Normal/Control': continue 
    valid_data = grouped_df.dropna(subset=['Months'])
    if len(valid_data) > 0:
        kmf.fit(valid_data['Months'], valid_data['Event'], label=name)
        kmf.plot_survival_function()

plt.title("Survival Rate: NSCLC vs SCLC (GSE30219)")
plt.xlabel("Months")
plt.ylabel("Survival Probability")
plt.grid(True)
plt.show()

# 데이터 확인용 (인원수)
print(df_target['Group'].value_counts())

## boxplot

In [ ]:
# SCLC, NSCLC간 생존율 차이 
survival_clc = df_target.groupby('Group').mean('Month').sort_values(by = 'Months', ascending = False)
plot_df = df_target[df_target['Group'] != 'Normal/Control']
order = ['NSCLC (Non-Small Cell)', 'Other', 'SCLC (Small Cell)']

plt.figure(figsize = (10, 6))
plt.title('Survival Time Distribution by Lung Cancer Type')
plt.xlabel('Group')
plt.ylabel('Months')
sns.boxplot(x = 'Group', y = 'Months', data = df_target, hue = 'Group', palette = 'deep', order = order)
plt.show()

print("--- [최종 분석] 주요 암종별 평균 생존 기간 ---")
print(df_target.groupby('Group')['Months'].mean().sort_values(ascending=False))

## 통계분석 (평균 생존 기간)

### Kruskal–Wallis test
- 일원분산분석(ANOVA)의 비모수 대응 방법

In [ ]:
# 숫자로우! 
df_target['Months'] = pd.to_numeric(df_target['Months'], errors='coerce')

# 그룹화
NSCLC_group = df_target[df_target['Group'] == "NSCLC (Non-Small Cell)"]
SCLC_group  = df_target[df_target['Group'] == "SCLC (Small Cell)"]
Other_group = df_target[~df_target['Group'].isin(["NSCLC (Non-Small Cell)","SCLC (Small Cell)"])]

# 결측값 제거 (안하니까 분석이 안됩니다... )
NSCLC_times = NSCLC_group['Months'].astype(float).dropna().values
SCLC_times  = SCLC_group['Months'].astype(float).dropna().values
Other_times = Other_group['Months'].astype(float).dropna().values

In [ ]:
stat, p = kruskal(NSCLC_times, SCLC_times, Other_times)

print(f"Kruskal-Wallis Test Statistic: {stat:.4f}")
print(f"p-value: {p:.4e}")

### 후속 분석-Dunn test
- 아, 튜키는 쟤랑 안 논대요. 

In [ ]:
# 이럴거면 변환 왜한겨
NSCLC_times = pd.Series(NSCLC_times, name='Months')
SCLC_times  = pd.Series(SCLC_times, name='Months')
Other_times = pd.Series(Other_times, name='Months')

# 데이터 준비
posthoc_df = pd.DataFrame({
    'Months': pd.concat([NSCLC_times, SCLC_times, Other_times]),
    'Group': (['NSCLC'] * len(NSCLC_times) +
              ['SCLC'] * len(SCLC_times) +
              ['Other'] * len(Other_times))
})


In [ ]:
dunn = sp.posthoc_dunn(
    posthoc_df,
    val_col='Months',
    group_col='Group',
    p_adjust='bonferroni'
)

print(dunn)

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(dunn, annot=True, fmt=".4f", cmap="Greys_r", cbar_kws={'label': 'Adjusted p-value'}
)

plt.title("Dunn Post-hoc Test (Bonferroni-adjusted)")
plt.tight_layout()
plt.show()

### SCLC, NSCLC만 맨 휘트니 해봤습니다. 

#### 왜 맨 휘트니죠? 를 설명하기 위한 사전 절차

In [ ]:
# 샤피로-윌크 검정
statistic, p_value = stats.shapiro(NSCLC_times)

print(f"--- Shapiro-Wilk Test ---")
print(f"검정 통계량(Statistic): {statistic:.4f}")
print(f"p-value: {p_value:.4f}")

# 해석 결과 출력
if p_value > 0.05:
    print("결과: 귀무가설을 기각하지 못함 (정규분포를 따른다고 볼 수 있음)")
else:
    print("결과: 귀무가설 기각 (정규분포를 따른다고 보기 어려움)")
print("\n")

In [ ]:
# 샤피로 검정 수행
statistic, p_value = stats.shapiro(SCLC_times)

print(f"--- Shapiro-Wilk Test ---")
print(f"검정 통계량(Statistic): {statistic:.4f}")
print(f"p-value: {p_value:.4f}")

# 해석 결과 출력
if p_value > 0.05:
    print("결과: 귀무가설을 기각하지 못함 (정규분포를 따른다고 볼 수 있음)")
else:
    print("결과: 귀무가설 기각 (정규분포를 따른다고 보기 어려움)")
print("\n")

In [ ]:
# 샤피로 검정 수행
statistic, p_value = stats.shapiro(Other_times)

print(f"--- Shapiro-Wilk Test ---")
print(f"검정 통계량(Statistic): {statistic:.4f}")
print(f"p-value: {p_value:.4f}")

# 해석 결과 출력
if p_value > 0.05:
    print("결과: 귀무가설을 기각하지 못함 (정규분포를 따른다고 볼 수 있음)")
else:
    print("결과: 귀무가설 기각 (정규분포를 따른다고 보기 어려움)")
print("\n")

- 두개가 정규분포를 따르지 아니하므로 맨 휘트니 U 검정을 해야 합니다. 

In [ ]:
# 레빈 검정
stat, p = levene(NSCLC_times, SCLC_times, Other_times)

print(f"Levene 통계량: {stat:.4f}, p-value: {p:.4f}")
if p < 0.05:
    print("등분산 아님 (귀무가설 기각)")
else:
    print("등분산임 (귀무가설 채택)")

- 등분산도 아닙니다. 맨 휘트니가 답입니다. 

In [ ]:
u_stat, p_value = mannwhitneyu(NSCLC_times, SCLC_times, alternative='two-sided')

print(f"U statistic: {u_stat:.4f}")
print(f"p-value: {p_value:.4e}")

In [ ]:
n1 = len(NSCLC_times)
n2 = len(SCLC_times)

mean_u = n1 * n2 / 2
std_u = np.sqrt(n1 * n2 * (n1 + n2 + 1) / 12)

z = (u_stat - mean_u) / std_u
r = abs(z) / np.sqrt(n1 + n2)

print(f"Effect size r: {r:.4f}")

- 일단 히트맵이 너무 압도적으로 까맣죠? 저게 어떤 의미냐... 일단 흰색 영역(주대각선)은 버리시면 됩니다. 저게 P-value라서 쉽게 말하자면 '쟤네들 차이가 통계적으로 유의하다'는 얘기예요. 
- 맨 휘트니 역시 소세포성 폐암과 비소세포성 폐암의 평균 생존기간에 차이가 있다는 걸 얘기하는겁니다. 
- 저 이펙트 사이즈는 뭔데요? 생존 기간을 결정하는 데 암종 말고 다른 요인들이 더 있다는 얘기죠. 

# 암종별 생존 곡선

In [ ]:
# 암종별 생존곡선 그리기
# 위에 그거...같은데? 
df_study = df_mut[[
    'characteristics_ch1.3.histology',
    'characteristics_ch1.7.follow-up time (months)',
    'characteristics_ch1.8.status'
]].copy()
df_study.columns = ['Histology', 'Months', 'Status']

# 전처리: 숫자 변환 및 사망 이벤트 정의
df_study['Months'] = pd.to_numeric(df_study['Months'], errors='coerce')
# 위랑 같음. 사망->1, 살아있음->0
df_study['Event'] = df_study['Status'].apply(lambda x: 1 if str(x).upper() == 'DEAD' else 0)

# 암종별 비교 
# ADC: 선암
# SQC: 편평상피세포암
# NTL: 정상 폐
# BAS: 기저세포양 편평세포암
# LCC: 대세포암
# LCNE: 대세포 신경내분비암
# SCC: 소세포암
# CARCI: 유암종
# Other: 기타
plt.figure(figsize=(12, 8))
ax = plt.subplot(111)
kmf = KaplanMeierFitter()

# 정상 제외하고 분석 
groups = df_study[df_study['Histology'] != 'NTL']['Histology'].unique()

for name in groups:
    group_data = df_study[df_study['Histology'] == name].dropna(subset=['Months'])
    if len(group_data) > 5: # 샘플 수가 너무 적은 그룹 제외
        kmf.fit(group_data['Months'], group_data['Event'], label=name)
        kmf.plot_survival_function(ax=ax)

plt.title("Lung Cancer Survival by Histology (GSE30219)", fontsize=15)
plt.xlabel("Months")
plt.ylabel("Survival Probability")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

# 인원수 및 사망자 확인
print("--- 그룹별 현황 ---")
print(df_study.groupby('Histology')['Event'].agg(['count', 'sum']).rename(columns={'sum': 'deaths'}))

## Boxplot

In [ ]:
# 암종별 평균 생존률(막대그래애프)
survival_stats = df_study.groupby('Histology')['Months'].mean().sort_values(ascending=False)
plot_df = df_study[df_study['Histology'] != 'NTL']
histology_order = ['CARCI', 'ADC', 'SQC', 'BAS', 'LCC', 'LCNE', 'SCC', 'Other']

sns.boxplot(x='Histology', y='Months', data=df_study, hue = 'Histology', palette='deep', order=histology_order)
plt.title('Average Survival Months by Histology')
plt.xlabel('Histology')
plt.ylabel('Average Survival Months')
plt.tight_layout()
plt.show()

print("--- [최종 분석] 주요 암종별 평균 생존 기간 ---")
print(df_study.groupby('Histology')['Months'].mean().sort_values(ascending=False))